# Compose Figure 4 with UMAP panel (e)

Stitches `Figure4.pdf` + `umap_by_coarse_group.pdf` into a single multi-panel PDF.
- Vectors preserved (no rasterization).
- Figure 4 is clipped to its actual content region (no whitespace).
- UMAP suptitle **and** the original above-plot column labels are clipped out.
- Column labels (MorphPT / Frozen DINOv3 / Frozen SimCLR) are redrawn **below** the plots so they sit closer to Figure 4.
- Panel `e` label added in matching style.

In [ ]:
# If PyMuPDF isn't available in your env:
# !pip install --quiet pymupdf

In [ ]:
import os
import fitz  # PyMuPDF

# ----- Paths -----
FIG_DIR = "/hpc/group/jilab/tc459/MorphPT/results/figures/feature_space_core500_emb_2p5x_fixedcolor"
FIG4_PDF = os.path.join(FIG_DIR, "Figure4.pdf")
UMAP_PDF = os.path.join(FIG_DIR, "umap_by_coarse_group.pdf")
OUT_PDF  = os.path.join(FIG_DIR, "Figure4_with_UMAP.pdf")

In [ ]:
# ----- Layout parameters -----

# Figure 4 content region (PyMuPDF top-left origin).
FIG4_CONTENT = fitz.Rect(0, 0, 903, 531)

# UMAP source: 900 x 312 pt.
# Bands in original:
#   y≈7–20  suptitle  (drop)
#   y≈43–62  column labels above plots  (drop, redraw below)
#   y≈67–300 the three UMAP scatter plots  (keep)
UMAP_PLOTS_CLIP = fitz.Rect(0, 62, 900, 312)   # plots only

# Original x-centers of the three column labels (extracted from UMAP PDF text).
LABEL_CENTERS_SRC_X = {
    "MorphPT":       151.2,
    "Frozen DINOv3": 450.0,
    "Frozen SimCLR": 748.8,
}

# Vertical gap between Figure 4 and panel e (room for the 'e' glyph).
GAP_LABEL = 28

# Spacing between plots bottom and the redrawn column labels.
GAP_TO_LABELS_BELOW = 8
# Bottom padding under labels.
BOTTOM_PAD = 6

# Font sizes.
PANEL_LABEL = "e"
PANEL_LABEL_FONTSIZE = 22       # larger panel 'e'
PANEL_LABEL_FONT     = "helvetica-bold"
PANEL_LABEL_X        = 6        # pt from left edge

COLUMN_LABEL_FONTSIZE = 16      # larger column labels
COLUMN_LABEL_FONT     = "helvetica"

In [ ]:
# ----- Compose -----
fig4_doc = fitz.open(FIG4_PDF)
umap_doc = fitz.open(UMAP_PDF)

# Match UMAP plot width to Figure 4's content width (uniform scale).
W = FIG4_CONTENT.width                # 903
H_FIG = FIG4_CONTENT.height           # 531
scale = W / UMAP_PLOTS_CLIP.width     # 903 / 900
plots_h = UMAP_PLOTS_CLIP.height * scale  # ~250.8 pt

# Total page height = Figure 4 + gap + plots + gap + labels + bottom pad.
total_h = (
    H_FIG
    + GAP_LABEL
    + plots_h
    + GAP_TO_LABELS_BELOW
    + COLUMN_LABEL_FONTSIZE
    + BOTTOM_PAD
)

out = fitz.open()
page = out.new_page(width=W, height=total_h)

# Place Figure 4 at top, clipped to content.
page.show_pdf_page(
    fitz.Rect(0, 0, W, H_FIG),
    fig4_doc, 0,
    clip=FIG4_CONTENT,
)

# Place UMAP plots below the gap.
y_plots_top = H_FIG + GAP_LABEL
y_plots_bot = y_plots_top + plots_h
page.show_pdf_page(
    fitz.Rect(0, y_plots_top, W, y_plots_bot),
    umap_doc, 0,
    clip=UMAP_PLOTS_CLIP,
)

# Panel 'e' label: vertically centered in the gap above the plots.
page.insert_text(
    fitz.Point(PANEL_LABEL_X, H_FIG + PANEL_LABEL_FONTSIZE + 2),
    PANEL_LABEL,
    fontname=PANEL_LABEL_FONT,
    fontsize=PANEL_LABEL_FONTSIZE,
)

# Redraw column labels below the plots, centered at the scaled source x-centers.
y_label_baseline = y_plots_bot + GAP_TO_LABELS_BELOW + COLUMN_LABEL_FONTSIZE
for text, src_cx in LABEL_CENTERS_SRC_X.items():
    dst_cx = src_cx * scale
    text_w = fitz.get_text_length(text, fontname=COLUMN_LABEL_FONT,
                                  fontsize=COLUMN_LABEL_FONTSIZE)
    x = dst_cx - text_w / 2
    page.insert_text(
        fitz.Point(x, y_label_baseline),
        text,
        fontname=COLUMN_LABEL_FONT,
        fontsize=COLUMN_LABEL_FONTSIZE,
    )

out.save(OUT_PDF)
print(f"Wrote {OUT_PDF}")
print(f"Page size: {W:.0f} x {total_h:.0f} pt  ({W/72:.2f} x {total_h/72:.2f} in)")

In [ ]:
# ----- Preview inline (optional) -----
from IPython.display import Image, display
doc = fitz.open(OUT_PDF)
pix = doc[0].get_pixmap(matrix=fitz.Matrix(1.2, 1.2))
display(Image(data=pix.tobytes("png")))